## Task 1 and 2 
* Add a Python script (CLI) that runs end-to-end: load → preprocess → train → evaluate → save artifacts, using arguments for input/output paths.
* Add support for running multiple model configurations in one execution and saving each run’s outputs in separate folders (metrics + artifacts + settings).

In [1]:
import sys
sys.argv = [
    "train_pipeline.py",
    "--train-data", "data/dataset_C_train.csv",
    "--val-data", "data/dataset_C_val.csv",
    "--output-dir", "runs",
    "--threshold-scoring", "f1",
    # "--configs", "configs.json"  # optional
]

import argparse
import json
import re
import warnings
from datetime import datetime
from pathlib import Path

import joblib
import nltk
import numpy as np
import pandas as pd

from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")

try:
    nltk.data.find("corpora/wordnet")
except LookupError:
    nltk.download("wordnet", quiet=True)

In [2]:
class TextPreprocessor:
    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()

    def lemmatize_text(self, text):
        if pd.isna(text) or text == "":
            return ""
        text = re.sub(r"[^\w\s]", " ", str(text).lower())
        return " ".join(
            self.lemmatizer.lemmatize(w)
            for w in text.split()
            if w.isalpha()
        )

    def preprocess_dataset(self, texts):
        return [self.lemmatize_text(t) for t in texts if self.lemmatize_text(t)]


In [3]:
class ModelTrainer:
    def __init__(self, model_type="logistic", model_params=None):
        self.model_type = model_type
        self.model_params = model_params or {}
        self.preprocessor = TextPreprocessor()
        self.vectorizer = None
        self.model = None

    def create_model(self):
        if self.model_type == "logistic":
            base = LogisticRegression(max_iter=1000, random_state=42, **self.model_params)
        elif self.model_type == "random_forest":
            base = RandomForestClassifier(random_state=42, **self.model_params)
        else:
            raise ValueError(f"Unknown model type: {self.model_type}")
        return MultiOutputClassifier(base)

    def train(self, X_train, y_train, X_val, y_val):
        X_train_clean = self.preprocessor.preprocess_dataset(X_train)

        self.vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=30000,
        )

        X_train_vec = self.vectorizer.fit_transform(X_train_clean)
        self.model = self.create_model()
        self.model.fit(X_train_vec, y_train)

        return self.evaluate(X_val, y_val)

    def evaluate(self, X, y_true):
        X_clean = self.preprocessor.preprocess_dataset(X)
        X_vec = self.vectorizer.transform(X_clean)

        y_pred = self.model.predict(X_vec)
        y_proba = self.model.predict_proba(X_vec)

        metrics = {}
        for i in range(y_true.shape[1]):
            metrics[f"label_{i+1}"] = {
                "accuracy": accuracy_score(y_true[:, i], y_pred[:, i]),
                "precision": precision_score(y_true[:, i], y_pred[:, i], zero_division=0),
                "recall": recall_score(y_true[:, i], y_pred[:, i], zero_division=0),
                "f1": f1_score(y_true[:, i], y_pred[:, i], zero_division=0),
            }

        return {
            "metrics": metrics,
            "predictions": y_pred,
            "probabilities": y_proba,
        }

In [4]:
class RunManager:
    def __init__(self, base_output_dir):
        self.base = Path(base_output_dir)
        self.base.mkdir(exist_ok=True)

    def create_run_dir(self, name):
        run_dir = self.base / name
        run_dir.mkdir(exist_ok=True)
        return run_dir

    def save(self, run_dir, model, vectorizer, preprocessor, results, config):
        joblib.dump(model, run_dir / "model.pkl")
        joblib.dump(vectorizer, run_dir / "vectorizer.pkl")
        joblib.dump(preprocessor, run_dir / "preprocessor.pkl")

        with open(run_dir / "metrics.json", "w") as f:
            json.dump(results["metrics"], f, indent=2)

        with open(run_dir / "config.json", "w") as f:
            json.dump(config, f, indent=2)

In [5]:
def main():
    parser = argparse.ArgumentParser(description="Multi-Label Text Classification Pipeline")
    parser.add_argument("--train-data", required=True)
    parser.add_argument("--val-data", required=True)
    parser.add_argument("--output-dir", default="runs")
    parser.add_argument("--threshold-scoring", default="f1")
    parser.add_argument("--configs", help="JSON file with multiple model configurations")

    args = parser.parse_args()

    train_df = pd.read_csv(args.train_data)
    val_df = pd.read_csv(args.val_data)

    X_train = train_df["text"].values
    y_train = train_df.drop("text", axis=1).values
    X_val = val_df["text"].values
    y_val = val_df.drop("text", axis=1).values

    label_names = [c for c in train_df.columns if c != "text"]

    if args.configs:
        with open(args.configs) as f:
            configurations = json.load(f)
    else:
        configurations = [{
            "name": "default_logistic",
            "model_type": "logistic",
            "model_params": {}
        }]

    manager = RunManager(args.output_dir)

    for config in configurations:
        print(f"\nTraining configuration: {config['name']}")

        trainer = ModelTrainer(
            model_type=config["model_type"],
            model_params=config.get("model_params", {})
        )

        results = trainer.train(X_train, y_train, X_val, y_val)

        run_dir = manager.create_run_dir(config["name"])

        run_config = {
            "train_data": args.train_data,
            "val_data": args.val_data,
            "model_type": config["model_type"],
            "model_params": config.get("model_params", {}),
            "labels": label_names,
            "timestamp": datetime.now().isoformat()
        }

        manager.save(
            run_dir,
            trainer.model,
            trainer.vectorizer,
            trainer.preprocessor,
            results,
            run_config
        )

        print(f"Saved run to {run_dir}")

if __name__ == "__main__":
    main()



Training configuration: default_logistic
Saved run to runs/default_logistic
